<a href="https://colab.research.google.com/github/evgenykomarov/sdc_course/blob/spring2026/seminar06-prediction-planning/ysda_homework_ppo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scenario Data Loading

This tutorial demonstrates how to load scenario data from the Waymo Open Motion Dataset (WOMD) using the Waymax dataloader.

In [1]:
# !pip install --upgrade pip
# !pip install git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax
# !pip install matplotlib==3.8.0

Необходимые для семинара данные и код лежат по ссылке https://drive.google.com/drive/folders/1iI_1PIFNx6-5MUIQkjslg1MqRhP7mikM?usp=sharing

Необходимо создать ярлык на своем Google Drive, как в прошлой дз, если вы еще этого не сделали

Монтируем гугл диск в локальную файловую систему

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

Ниже нужно указать путь до созданного ранее ярлыка, например, если ярлык имеет путь `ai360`, то должно получиться
```
SEMINAR_PATH = '/content/drive/MyDrive/ai360'
```

In [3]:
# SEMINAR_PATH = '/content/drive/MyDrive/ysda-prediction'
SEMINAR_PATH = './data'

In [4]:
import os
import shutil

# if not os.path.exists('lib'):
#     shutil.copytree(os.path.join(SEMINAR_PATH, 'lib'), 'lib')
# else:
#     print('"lib" folder already exists. If you want to rewrite lib by original folder, remove local "lib" manually')

In [5]:
# %%capture

import os
from copy import deepcopy

import numpy as np
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F

import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar

from waymax.config import DatasetConfig

## you need to import code for PlanningModel and NormalizeSceneWrapper classes from your previous homework
# from <your path to code with PlanningModel class here> import PlanningModel, NormalizeSceneWrapper
from planning_model import PlanningModel, get_notebook_name, load_planning_from_ckpt, SmartNormalizeSceneWrapper as NormalizeSceneWrapper
from lib.data_utils import WaymaxDataset, scenario_to_features_gt

device = 'cuda'

## Здесь вам нужно загрузить веса своей обученной модели планера из предыдущего дз

In [6]:
# """your code here"""

# checkpoint = torch.load("path to your checkpoint", map_location='cuda')
# model_state_dict = {k.replace('model.model.', 'model.'): v for k, v in checkpoint['state_dict'].items()
#                     if k.startswith('model.model')}
# # Then load into your standalone model
# model = NormalizeSceneWrapper(PlanningModel())
# model.load_state_dict(model_state_dict)

ckpt_path = './checkpoint_planning/best-epoch=19-val/epoch/min_ade=0.41.ckpt'
model = load_planning_from_ckpt(ckpt_path)
max_num_objects = 12

In [7]:
def get_data_config(split_name, seminar_path=SEMINAR_PATH, max_num_objects=24):
    split_path = os.path.join(SEMINAR_PATH, 'data', split_name)

    obj_count = int(os.listdir(split_path)[0].rsplit('-')[-1])
    return DatasetConfig(
        path=os.path.join(split_path, f'{split_name}_tfexample.tfrecord@{obj_count}'),
        max_num_objects=max_num_objects,
        batch_dims=[8],
        repeat=1,
        shuffle_buffer_size=64,
        deterministic=False,
        num_shards=1
    )

In [8]:
train_dataset = WaymaxDataset(get_data_config('training', max_num_objects=max_num_objects))
val_dataset = WaymaxDataset(get_data_config('validation', max_num_objects=max_num_objects))

In [9]:
FUTURE_STEPS = 30

# Дообучение модели планнера с помощью RL

В данном домашнем задании вам предстоит реализовать алгоритм PPO - Proximal Policy Optimization для дообучения модели планнинга. Для того чтобы ознакомиться с идеей алгоритма, рекомендую прочесть полную статью https://arxiv.org/pdf/1707.06347 или краткую сводку с всей нужной информацией https://spinningup.openai.com/en/latest/algorithms/ppo.html

In [10]:
from waymax import config as _config
from waymax import dataloader
from waymax import datatypes
from waymax import dynamics
from waymax import env as _env
from waymax import agents
from waymax import visualization
from waymax.agents import SimAgentActor, WaymaxActorOutput
from waymax import metrics

from torch.distributions import Categorical

import jax
from jax import numpy as jnp
from jax import random

import chex

import dataclasses

from typing import Any, Optional, Callable, Sequence
import mediapy

# PPO Algorithm

## Initializing the ego-agent as RL agent

**Класс EgoAgent** является классом, который управляет эго-агентом в симуляции. Он наследуется от класса SimAgentActor из Waymax и расширяет его функциональность для работы с моделью планнера, которая предсказывает следующие действия агента.


1. Имплементация метода select_action не отличается от базовой имплементации, кроме того, что мы добавляем поле log_probs в выход, чтобы во время обучения использовать их для подсчета ppo loss.


2. Метод update_trajectory является ключевым в классе EgoAgent. Он отвечает за выбор действия на основе текущей политики и возвращает логарифмические вероятности действий, которые используются для вычисления лосса в алгоритме PPO.

In [11]:
def extract_best_mode_from_pred_component(pred, best_mode):
    return pred[torch.arange(best_mode.shape[0]), best_mode]

def extract_first_timestep_from_pred(pred):
    return pred[..., 0]

In [12]:
ActorState = datatypes.PyTree
Params = datatypes.PyTree
Action = datatypes.PyTree


@chex.dataclass(frozen=True)
class RLActorOutput(agents.WaymaxActorOutput):
    """Output of the RL actor, extending WaymaxActorOutput with log_probs.

    Attributes:
        actor_state: Internal state for whatever the agent needs to keep as its
          state. This can be recurrent embeddings or accounting information.
        action: Action of shape (..., num_objects) predicted by the RL actor.
        is_controlled: A binary indicator of shape (..., num_objects) representing
          which objects are controlled by the actor.
        log_probs: Log_probs output by the RL actor, representing the log probability
          of policy's actions
    """

    log_probs: Optional[jax.Array]


class EgoAgent(SimAgentActor):
    def __init__(
        self,
        model,
        is_controlled_func: Optional[Callable[[datatypes.SimulatorState], jax.Array]] = None
    ):
        super().__init__(is_controlled_func=is_controlled_func)
        model.to(device)
        self.model = model


    def update_trajectory(
        self, state: datatypes.SimulatorState
    ) -> datatypes.TrajectoryUpdate:
        """Updates the trajectory for all simulated agents."""

        features, _ = scenario_to_features_gt(state, features_first_timestamp=state.timestep - 10, gt_timestamps=1, device=device)
        pred = self.model(features)
        logits = pred['logits']
        mode_distribution = Categorical(logits=logits)
        mode_sample = mode_distribution.sample()
        log_probs = mode_distribution.log_prob(mode_sample)

        trajectory_jax = jax.tree_util.tree_map(
            lambda x: jnp.repeat(
                jnp.array(
                    extract_first_timestep_from_pred(
                        extract_best_mode_from_pred_component(x, mode_sample)
                    ).cpu().detach().numpy()
                ).reshape(-1, 1, 1),
                state.log_trajectory.x.shape[-2],
                axis=-2
            ),
            pred['trajectory']
        )

        x = trajectory_jax['x']
        y = trajectory_jax['y']
        vel_x = trajectory_jax['vel_x']
        vel_y = trajectory_jax['vel_y']
        yaw = trajectory_jax['yaw']

        # predictions is valid only for sdc
        valid = jnp.bool_(jnp.zeros_like(x))
        valid = valid.at[state.object_metadata.is_sdc].set(True)

        return datatypes.TrajectoryUpdate(x=x, y=y, yaw=yaw, vel_x=vel_x, vel_y=vel_y, valid=valid), log_probs


    def select_action(
        self,
        params: Params,
        state: datatypes.SimulatorState,
        actor_state: Any,
        rng: jax.Array,
    ) -> agents.WaymaxActorOutput:
        """Selects action and updates trajectory given the current simulator state."""

        del actor_state, rng  # Not used
        action, log_probs = self.update_trajectory(state)
        action = action.as_action() # here we transform the action which we got from the model to a desired datatype [look above]

        return RLActorOutput(
            action=action,
            actor_state=None,
            is_controlled=self.is_controlled_func(state),
            log_probs=log_probs,
        )

    @property
    def name(self) -> str:
        return self.__class__.__name__

## Rewards, Reward-to-go computation and PPO Loss function

### Rewards {1.5 балла}

Ниже вам нужно будет создать различные функции для подсчета наград при помощи модуля waymax.rewards. Эти функции наград используются в симуляции для оценки поведения агентов в зависимости от их действий. Зайдите в репозиторий https://github.com/waymo-research/waymax/tree/main/waymax/metrics и поймите, как добавлять следующие реворды в обучение: imitation reward, offroad reward, overlap reward, comfort reward. Также, разберитесь в смысле каждого реворда, исходя из кода waymax/metrics, и опишите их здесь:

1. **Log divergence:** < your explanation >

2. **Offroad reward:** < your explanation >

3. **Overlap reward:** < your explanation >

4. **Comfort reward:** < your explanation >

С помощью linear combination reward из модуля waymax мы можем совмещать несколько наград и считать суммарную награду на каждом шаге симуляции. Давайте для начала добавим log_divergence, offroad, overlap rewards

В конфиге награды можно менять величину штрафа. Например, если мы видим, что модель склонна часто выезжать за пределы дороги, можно увеличить штраф за offroad, чтобы модель с большей вероятностью отвергала действия, приводящие к выезду за пределы дороги.

Также, посмотрев в код наград, подумайте, с каким знаком их нужно добавлять в наше обучение. Здесь важно помнить, что алгоритмы RL максимизируют награду

In [13]:
from waymax.rewards import linear_combination_reward

"""
your code here
"""

imitation_config = _config.LinearCombinationRewardConfig({'log_divergence': -1.0})
imitation_reward_function = linear_combination_reward.LinearCombinationReward(imitation_config)

offroad_config = _config.LinearCombinationRewardConfig({'offroad': -1.0})
offroad_reward_function = linear_combination_reward.LinearCombinationReward(offroad_config)

overlap_config = _config.LinearCombinationRewardConfig({'overlap': -1.0})
overlap_reward_function = linear_combination_reward.LinearCombinationReward(overlap_config)

all_rewards_config = _config.LinearCombinationRewardConfig({
    'log_divergence': -1.0,
    'offroad': -2.0,
    'overlap': -5.0
})
combination_reward_function = linear_combination_reward.LinearCombinationReward(all_rewards_config)

#### Задания 1.1 и 1.2 {2 балла}:

1.1 **Имплементировать PPO-Clip Loss** {1.5 баллов}
PPO-Clip Loss используется для оптимизации политики агента. PPO-Clip обновляет веса с помощью градиентного подъема:
$$
\theta_{k+1} = \arg \max_{\theta} \mathbb{E}_{s,a \sim \pi_{\theta_k}} \left[ L(s, a, \theta_k, \theta) \right],
$$
L имеет вид:
$$
L(s, a, \theta_k, \theta) = \min\left(
    \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)} A^{\pi_{\theta_k}}(s, a), \;
    \text{clip}\left(
        \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)}, 1 - \epsilon, 1 + \epsilon
    \right) A^{\pi_{\theta_k}}(s, a)
\right),
$$

Epsilon - это clipping factor, контролирующий, насколько далеко от референсной политики $\pi_{\theta_k}$ может уйти обучаемая политика $\pi_\theta$.
Так как современные оптимайзеры решают задачу минимизации, а в PPO мы наоборот максимизируем ожидаемую награду, то при обучении будем домножать лосс на -1, чтобы градиентный спуск фактически поднимал reward.

В стандартном случае, $A(s, a) = Q(s, a) - V(s)$, где $V(s)$ - произвольная функция или модель, которая умеет оценивать бейзлайны, то есть среднюю награду, которую получит наш агент, начав в стейте s и придерживаясь политики $\pi_\theta$ до конца эпизода. Основная цель добавления $V(s)$ в подсчет advantage-ей - снизить дисперсию в оценке реворда. Для того чтобы не нагромождать эту дз, мы не будем обучать отдельную Value Model для оценки бейзлайнов, вместо этого будем считать, что $A(s, a) = normalized (Q(s, a))$, что так же позволяет снизить дисперсию.

1.2  **Имплементировать Reward-to-go** {0.5 балл}


$Q(s, a)$ - функция, оценивающая ожидаемый куммулятивный реворд от того, что агент принял action a в стейте s. Есть различные способы оценивать $Q(s, a)$, например - мгновенная награда от выполнения action a или сумма всех наград за эпизод. Чтобы оценить полезность action a в state s, агент должен ориентироваться на последствия от предпринятого action-a. Реворды, которые были получены до этого момента не показывают, насколько хорошее был действие. Соответственно, мы хотим считать $Q(s, a)$ как куммулятивную сумму будущих ревордов (reward-to-go)


Формула для Reward-to-Go:
$$ G_t = \sum_{t'=t}^{T} \gamma^{t'-t} \cdot r_{t'} $$
Где: $ r_{t'} $ — награда на шаге t′. $γ$ — коэффициент дисконтирования.



In [14]:
def compute_PPO_loss(advantages, ratio, clip_coef=0.1):
    """
    Implementation of ppo loss
    input:
    advantages.shape (bs, num_ticks)
    ratio.shape (bs, num_ticks)

    output:
    ppo_loss.shape (bs)
    """
    return torch.minimum(ratio * advantages, ratio.clip(min=1.0 - clip_coef, max=1.0 + clip_coef) * advantages).mean(axis=-1)

def reward2go(reward, gamma=0.99):
    """
    Compute reward-to-go: G_t = sum_{t'>=t} gamma^{t'-t} * r_{t'}
    input:
    reward.shape (bs, num_ticks)
    gamma - float
    output:
    rtgs.shape (bs, num_ticks)
    """
    _, num_ticks = reward.shape
    discount = gamma ** torch.arange(num_ticks, device=reward.device)
    discounted_reward = discount[None, ...] * reward
    cum_reward = discounted_reward.flip(dims=[1]).cumsum(dim=1).flip(dims=[1])
    return cum_reward


## Initializing the environment and actors around us

#### Задание 2.1 {1 балл}:
В объекте state.object_metadata содержится информация о всех агентах на сцене.
**Найдите в этом объекте маску, которая возвращает True для агентов, соответствующих беспилотному автомобилю (SDC).**

**Настройте управление агентами:**
Для актора IDM_actors настройте функцию is_controlled_func так, чтобы она возвращала True для всех агентов, которые не являются беспилотным автомобилем.
Для актора actor_ego настройте функцию is_controlled_func так, чтобы она возвращала True только для агентов, которые являются беспилотным автомобилем.

In [15]:
dynamics_model = dynamics.StateDynamics()

# Number of agents on the scene, can be changed if you have enough compute to take actions for more agents in your env
max_num_objects = 12

# env = _env.MultiAgentEnvironment(
env = _env.BaseEnvironment(
    dynamics_model=dynamics_model,
    config=dataclasses.replace(
        _config.EnvironmentConfig(),
        max_num_objects=max_num_objects,
        controlled_object=_config.ObjectType.VALID,
    ),
)

In [16]:
IDM_actors = agents.IDMRoutePolicy(
    is_controlled_func=lambda state: ~state['object_metadata']['is_sdc']
)  # these are intelligent driver models, see more here https://github.com/waymo-research/waymax/blob/main/waymax/agents/waypoint_following_agent.py#L200

actor_ego = EgoAgent(model, is_controlled_func=lambda state: state['object_metadata']['is_sdc'])

actors = [actor_ego, IDM_actors]

select_action_list = [actor.select_action for actor in actors]
step = env.step

# Train loop for PPO {5 баллов}

В данном задании вам предстоит заполнить пропуски в train loop-е для обучения алгоритма PPO.

In [17]:
class PPOModule(pl.LightningModule):
    """PPO training wrapped as a Lightning module.

    Uses manual optimization (automatic_optimization=False) so that each
    training_step can run `ppo_epochs` gradient updates on the same rollout,
    which is the standard PPO pattern.
    """

    def __init__(
        self,
        model,
        env,
        select_action_list,
        combination_reward_function,
        reward_functions: dict = None,
        gamma: float = 0.99,
        clip_epsilon: float = 0.2,
        ppo_epochs: int = 3,
        start_timestep: int = 11,
        lr: float = 1e-4,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=['model', 'env', 'select_action_list',
                                          'combination_reward_function', 'reward_functions'])
        self.model = model
        self.env = env
        self.select_action_list = select_action_list
        self.combination_reward_function = combination_reward_function
        # e.g. {'imitation': imitation_reward_function, 'offroad': offroad_reward_function, ...}
        self.reward_functions = reward_functions or {}
        self.gamma = gamma
        self.clip_epsilon = clip_epsilon
        self.ppo_epochs = ppo_epochs
        self.start_timestep = start_timestep
        self.lr = lr

        self.automatic_optimization = False  # manual opt for multi-step PPO update

    # ------------------------------------------------------------------
    # Rollout collection (no grad)
    # ------------------------------------------------------------------
    def _collect_rollout(self, scenario):
        """Run the reference policy for FUTURE_STEPS and collect transitions.

        Stores per-timestep features tensors so _ppo_update can re-run only
        the PyTorch model forward (no JAX trajectory reconstruction needed).

        Returns:
            features_list: list of feature dicts, length T  — for PPO re-forward
            old_log_probs: (bs, T)
            rewards: (bs, T)  — total combination reward
            component_rewards: dict[name -> (bs, T)]
        """
        features_list, old_log_probs_list, rewards_list = [], [], []
        component_lists = {name: [] for name in self.reward_functions}

        # map reward function name → raw metric name in waymax
        _name_to_metric = {'imitation': 'log_divergence', 'offroad': 'offroad', 'overlap': 'overlap'}

        # map reward function name → raw metric name in waymax
        _name_to_metric = {'imitation': 'log_divergence', 'offroad': 'offroad', 'overlap': 'overlap'}

        current_state = self.env.reset(scenario)
        for timestep in range(self.start_timestep, self.start_timestep + FUTURE_STEPS):
            with torch.no_grad():
                # extract features before select_action so we can reuse them in _ppo_update
                features, _ = scenario_to_features_gt(
                    current_state,
                    features_first_timestamp=current_state.timestep - 10,
                    gt_timestamps=1,
                    device=self.device,
                )
                pred = self.model(features)
                logits = pred['logits']
                mode_dist = Categorical(logits=logits)
                mode_sample = mode_dist.sample()
                log_probs = mode_dist.log_prob(mode_sample)

                # build the action the same way EgoAgent does
                trajectory_jax = jax.tree_util.tree_map(
                    lambda x: jnp.repeat(
                        jnp.array(
                            extract_first_timestep_from_pred(
                                extract_best_mode_from_pred_component(x, mode_sample)
                            ).cpu().detach().numpy()
                        ).reshape(-1, 1, 1),
                        current_state.log_trajectory.x.shape[-2],
                        axis=-2,
                    ),
                    pred['trajectory'],
                )
                traj_update = datatypes.TrajectoryUpdate(
                    x=trajectory_jax['x'],
                    y=trajectory_jax['y'],
                    yaw=trajectory_jax['yaw'],
                    vel_x=trajectory_jax['vel_x'],
                    vel_y=trajectory_jax['vel_y'],
                    valid=jnp.bool_(jnp.zeros_like(trajectory_jax['x'])).at[
                        current_state.object_metadata.is_sdc
                    ].set(True),
                )
                ego_action = traj_update.as_action()

                # IDM actors for non-ego agents
                idm_output = self.select_action_list[1](
                    {'timestep': timestep}, current_state, None, None
                )
                action = agents.merge_actions([
                    RLActorOutput(
                        action=ego_action,
                        actor_state=None,
                        is_controlled=current_state.object_metadata.is_sdc,
                        log_probs=None,
                    ),
                    idm_output,
                ])

            agent_mask = current_state.object_metadata.is_sdc
            total_reward = self.combination_reward_function.compute(
                current_state, action, agent_mask
            ).mean(axis=-1)
            reward = torch.tensor(np.asarray(total_reward), device=self.device)

            features_list.append(features)
            old_log_probs_list.append(log_probs)
            rewards_list.append(reward)

            # compute each component reward using its fn (already correctly signed)
            for name, fn in self.reward_functions.items():
                r = fn.compute(current_state, action, agent_mask).mean(axis=-1)
                component_lists[name].append(
                    torch.tensor(np.asarray(r), device=self.device, dtype=torch.float32)
                )

            current_state = self.env.step(current_state, action)

        rewards = torch.stack(rewards_list, dim=1)              # (bs, T)
        old_log_probs = torch.stack(old_log_probs_list, dim=1)  # (bs, T)
        component_rewards = {
            name: torch.stack(steps) for name, steps in component_lists.items()
        }
        return features_list, old_log_probs, rewards, component_rewards

    # ------------------------------------------------------------------
    # PPO update epochs
    # ------------------------------------------------------------------
    def _ppo_update(self, features_list, old_log_probs, normalized_rtgs):
        """Run `ppo_epochs` gradient steps.

        Re-runs only the PyTorch model forward (no JAX env interaction),
        which is the dominant speed improvement vs. calling select_action again.
        """
        opt = self.optimizers()
        policy_loss_val = None

        for _ in range(self.ppo_epochs):
            new_log_probs_list = []
            for features in features_list:
                pred = self.model(features)
                mode_dist = Categorical(logits=pred['logits'])
                # re-sample is intentional: we want log prob under new policy for
                # the *same* action distribution shape; importance ratio handles the rest
                mode_sample = mode_dist.sample()
                new_log_probs_list.append(mode_dist.log_prob(mode_sample))

            new_log_probs = torch.stack(new_log_probs_list, dim=1)  # (bs, T)
            ratios = torch.exp(new_log_probs - old_log_probs)
            policy_loss = -compute_PPO_loss(normalized_rtgs, ratios,
                                            clip_coef=self.clip_epsilon).mean()

            opt.zero_grad()
            self.manual_backward(policy_loss)
            self.clip_gradients(opt, gradient_clip_val=0.5, gradient_clip_algorithm='norm')
            opt.step()

            policy_loss_val = policy_loss.detach()

        return policy_loss_val

    # ------------------------------------------------------------------
    # Lightning hooks
    # ------------------------------------------------------------------
    def training_step(self, scenario, _batch_idx):
        states, old_log_probs, rewards, component_rewards = self._collect_rollout(scenario)

        rtgs = reward2go(rewards, gamma=self.gamma)
        normalized_rtgs = (rtgs - rtgs.mean(dim=-1, keepdim=True)) / (
            rtgs.std(dim=-1, keepdim=True) + 1e-8
        )

        policy_loss = self._ppo_update(states, old_log_probs, normalized_rtgs)

        bs = rewards.shape[0]
        self.log('train/epoch/policy_loss', policy_loss,      batch_size=bs, on_step=False, on_epoch=True)
        self.log('train/epoch/avg_reward',  rewards.mean(),   batch_size=bs, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train/step/policy_loss',  policy_loss,      batch_size=bs, on_step=True,  on_epoch=False)
        self.log('train/step/avg_reward',   rewards.mean(),   batch_size=bs, on_step=True,  on_epoch=False)

        for name, r in component_rewards.items():
            self.log(f'train/epoch/reward_{name}', r.mean(), batch_size=bs, on_step=False, on_epoch=True)
            self.log(f'train/step/reward_{name}',  r.mean(), batch_size=bs, on_step=True,  on_epoch=False)

        del rewards, rtgs, normalized_rtgs, old_log_probs, component_rewards
        torch.cuda.empty_cache()

        return policy_loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.trainer.max_epochs,
            eta_min=self.lr * 1e-3,
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'},
        }


In [18]:
run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_dir = f'checkpoints/run_{run_tag}'
os.makedirs(ckpt_dir, exist_ok=True)
# shutil.copy(__file__, os.path.join(ckpt_dir, 'prediction_model.py'))
shutil.copy(get_notebook_name(), os.path.join(ckpt_dir, 'ppo_model.ipynb'))


ppo_module = PPOModule(
    model=model,
    env=env,
    select_action_list=select_action_list,
    combination_reward_function=combination_reward_function,
    reward_functions={
        'imitation': imitation_reward_function,
        'offroad':   offroad_reward_function,
        'overlap':   overlap_reward_function,
    },
    gamma=0.99,
    clip_epsilon=0.2,
    ppo_epochs=1,
    start_timestep=11,
    lr=1e-4,
)

logger = TensorBoardLogger('tb_logs', name='ppo_model')

checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename='best-{epoch:02d}-{train/epoch/avg_reward:.4f}',
    monitor='train/epoch/avg_reward',
    mode='max',
    save_top_k=3,
    # every_n_epochs=1,
    every_n_train_steps=3,
)

trainer = pl.Trainer(
    logger=logger,
    log_every_n_steps=1,
    max_epochs=2,
    accelerator=device,
    callbacks=[checkpoint_callback, TQDMProgressBar()],
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(ppo_module, train_dataloaders=train_dataset)

/home/evgenykomarov/.pyenv/versions/3.12.13/envs/waymax_fresh/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/evgenykomarov/jupyter_notebooks/sdc_course/seminar06-prediction-planning/checkpoints/run_20260605_133845 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type             ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ PredictionModule │  529 K │ eval │     0 │
└───┴───────┴──────────────────┴────────┴──────┴───────┘

Trainable params: 529 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 529 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 0                                                                                           
Modules in eval mode: 82                                                                                           
Total FLOPs: 0

I0000 00:00:1780655925.756207  562851 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31126 MB memory:  -> device: 0, name: Tesla V100-PCIE-32GB, pci bus id: 0000:81:00.0, compute capability: 7.0


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: (<gast.gast.NamedExpr object at 0x7f12847c6f90>, (leaf_jax_array := getattr(leaf, '__jax_array__', None)))
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: (<gast.gast.NamedExpr object at 0x7f12847c6f90>, (leaf_jax_array := getattr(leaf, '__jax_array__', None)))
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Pl

/home/evgenykomarov/.pyenv/versions/3.12.13/envs/waymax_fresh/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:534: Found 82 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |                                                                                                   …

### Можете остановить обучение, когда будете достигать Avg Step Reward = -0.15 или когда у вас кончится квота в колабе:)

# Проверим обученную модель на валидационном датасете

Так как делать шаги в симуляторе довольно дорого и долго, мы прогоним на валидации только часть датасета, чтобы не забирать у вас всю гпу квоту. Если у вас заканчивается квота, можете уменьшить количество сценариев

In [37]:
def evaluation(dataset, history_size=11, log_interval=1, num_scenarios=5):
    metrics = {
        'total': [],
        'distance': [],
        'offroad': [],
        'overlap': []
    }

    for i, scenario in enumerate(dataset, 1):
        episode_metrics = {k: [] for k in metrics}
        state = env.reset(scenario)

        for timestep in range(history_size, history_size + FUTURE_STEPS):
            with torch.no_grad():
                outputs = [select_action({'timestep': timestep}, state, None, None)
                         for select_action in select_action_list]
                action = agents.merge_actions(outputs)
                mask = state.object_metadata.is_sdc

            # Compute all rewards at once
            rewards = {
                'total': combination_reward_function.compute(state, action, mask),
                'distance': imitation_reward_function.compute(state, action, mask),
                'offroad': offroad_reward_function.compute(state, action, mask),
                'overlap': overlap_reward_function.compute(state, action, mask)
            }

            # Store rewards
            for k in episode_metrics:
                episode_metrics[k].append(torch.tensor(np.asarray(rewards[k]), device=device).mean(axis=-1))

            state = step(state, action)

        # Stack and store episode results
        for k in metrics:
            metrics[k].append(torch.stack(episode_metrics[k], dim=1))

        # Periodic logging
        if i % log_interval == 0:
            print(f"Validation Metrics, step: {i}")
            for k, v in metrics.items():
                mean_reward = torch.mean(torch.cat([m.mean(dim=1) for m in v]))
                print(f"Mean Episode {k.capitalize()} Reward: {mean_reward.item():.2f}")
        if i == num_scenarios:
            break

evaluation(val_dataset)

Validation Metrics, step: 1
Mean Episode Total Reward: 0.03
Mean Episode Distance Reward: 0.03
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 2
Mean Episode Total Reward: 0.03
Mean Episode Distance Reward: 0.03
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 3
Mean Episode Total Reward: 0.04
Mean Episode Distance Reward: 0.04
Mean Episode Offroad Reward: -0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 4
Mean Episode Total Reward: 0.03
Mean Episode Distance Reward: 0.04
Mean Episode Offroad Reward: -0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 5
Mean Episode Total Reward: 0.04
Mean Episode Distance Reward: 0.04
Mean Episode Offroad Reward: -0.00
Mean Episode Overlap Reward: 0.00


# Rollout generation - генерация видосиков {0.5 балла}
Скачайте несколько роллаутов и приложите их вместе с домашним заданием

In [42]:
def generate_close_loop(model, scenario, steps=None):
    dynamics_model = dynamics.StateDynamics()

    max_num_objects = 12

    # Environment to control all objects on scene
    # env = _env.MultiAgentEnvironment(
    env = _env.BaseEnvironment(
        dynamics_model=dynamics_model,
        config=dataclasses.replace(
            _config.EnvironmentConfig(),
            max_num_objects=max_num_objects,
            controlled_object=_config.ObjectType.VALID,
        ),
    )

    state = env.reset(scenario)

    # intelligent driver model actor for non-ego objects
    IDM_actors = agents.IDMRoutePolicy(
        is_controlled_func=lambda state: state.object_metadata.is_sdc == False
    )

    # our model actor for sdc
    actor_ego = EgoAgent(
        model,
        is_controlled_func=lambda state: state.object_metadata.is_sdc == True
    )

    actors = [actor_ego, IDM_actors]
    select_action_list = [actor.select_action for actor in actors]

    states = [env.reset(scenario)]

    if steps is None:
        steps = states[0].remaining_timesteps

    for timestep in range(0, steps):
        current_state = states[-1]
        outputs = [
            select_action({'timestep': timestep}, current_state, None, None)
            for select_action in select_action_list
        ]
        # make action for all objects on scene
        action = agents.merge_actions(outputs)

        # make step
        next_state = env.step(current_state, action)
        states.append(next_state)

    return states[1:]

In [43]:
from collections import defaultdict
from waymax import metrics

def plot_states(states, use_log_traj=False, batch_idx=0):
    imgs = []
    for state in states:
        imgs.append(visualization.plot_simulator_state(
            state, use_log_traj=use_log_traj, batch_idx=batch_idx))
    mediapy.show_video(imgs, fps=10)

def get_ego_metrics(states, start_index=0):
    metrics_config = _config.MetricsConfig()

    metrics_per_time = defaultdict(list)
    for state in states[start_index:]:
        all_metrics = metrics.run_metrics(state, metrics_config)
        for k, v in all_metrics.items():
            metrics_per_time[k].append(np.asarray(v.value[state.object_metadata.is_sdc]))

    return {
        k: np.mean(metrics_per_time[k], axis=0)
        for k in metrics_per_time.keys()
    }

In [89]:
scenario = next(iter(val_dataset))
# scenario = next(iter(train_dataset))

In [90]:
states = generate_close_loop(model, scenario, steps=50)
print(get_ego_metrics(states))

{'log_divergence': array([0.20506196, 0.14716168, 0.16021621, 0.49411488, 0.17356995,
       0.13576895, 0.28930175, 1.113791  ], dtype=float32), 'overlap': array([0. , 0. , 0. , 0. , 0. , 0. , 0.3, 0. ], dtype=float32), 'offroad': array([0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)}


In [91]:
plot_states(states, batch_idx=0)

This browser does not support the video tag.


In [92]:
plot_states(states, batch_idx=1)

This browser does not support the video tag.


In [93]:
plot_states(states, batch_idx=2)

This browser does not support the video tag.


In [94]:
plot_states(states, batch_idx=3)

This browser does not support the video tag.


In [95]:
plot_states(states, batch_idx=4)

This browser does not support the video tag.


In [96]:
plot_states(states, batch_idx=5)

This browser does not support the video tag.


In [97]:
plot_states(states, batch_idx=6)

This browser does not support the video tag.


In [98]:
plot_states(states, batch_idx=7)

This browser does not support the video tag.
